# TTM end to end, entirely through `run_recipe`

The full lifecycle for adapting a foundation model to one asset, with **no code below the tool
surface** - no `resolve()`, no `fit()`, no `save_pretrained()`. Every step is a tool call:

```
run_recipe (zero-shot baseline)
run_recipe (finetune + save_to)  ->  a checkpoint on disk
register_finetuned               ->  a catalog card pointing at it, with lineage
run_recipe (serve the new card)  ->  forecasts from YOUR fine-tuned weights
run_recipe (compare)             ->  did the fine-tune actually help?
list_runs                        ->  the whole thing is reproducible from the ledger
```

This is the loop that used to require dropping to raw sktime. As of the finetune-lifecycle fix, the
`finetune` block reaches the estimator and `save_to` persists the weights, so `run_recipe` closes it.

---

### Provenance

**Cells that touch torch/HuggingFace were not executed by the author** (no HF access in the authoring
sandbox) and are marked. Everything else - `recipe_template`, card registration, `resolve_model`
preflight, `register_finetuned`, `list_runs`, the regime logic - **was executed against the patched
server on this branch** and carries real output. Those calls degrade with an actionable message when
torch is missing rather than crashing, so the notebook runs top to bottom either way.

## 1. Setup

In [ ]:
import importlib.util, subprocess, sys
need = [p for p in ("torch", "transformers", "accelerate") if importlib.util.find_spec(p) is None]
if need:
    print("installing:", need, "- pulls the CUDA stack, several GB, a few minutes")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "torch", "transformers", "accelerate>=0.26.0"])
    print("done - RESTART THE KERNEL, then re-run")
else:
    print("torch / transformers / accelerate present")

In [ ]:
import os, sys, json, time, shutil, warnings
warnings.filterwarnings("ignore")
SRC = os.path.abspath("src")
os.environ["PYTHONPATH"] = SRC + os.pathsep + os.environ.get("PYTHONPATH", "")
os.environ["TSFM_STORE"] = "memory"          # swap for couch to use the real catalog
sys.path.insert(0, SRC)

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from servers.tsfm import main as M
from servers.tsfm.io import refs

CKPT = os.path.abspath("artifacts/ttm_r2_chiller6_ft")   # where the fine-tuned weights will land
print("store:", type(M._STORE).__name__)
print("checkpoint dir:", CKPT)

## 2. The asset: a chiller with 30 days of history

TTM-R2's stock checkpoint is **context 512 / prediction 96**, so `fit()` needs at least 512 points
to form one window. "Tiny" for a foundation model means **hundreds** of points, not tens - tiny
relative to the ~700M samples TTM was pretrained on. With 40 points you would not fine-tune at all;
you would use TTM zero-shot, which is the whole pitch.

Our asset: commissioned 30 days ago, 720 hourly points. It runs hotter and swings harder than the
pretrained fleet distribution - the kind of local deviation a fine-tune can pick up. We hold out the
last 24 hours to score on.

In [ ]:
SP, N, HOLD = 24, 720, 24
FH = list(range(1, 13))          # forecast 12h ahead
rng = np.random.RandomState(0)
t = np.arange(N)
load = (26.0 + 7.0*np.sin(t/SP*2*np.pi) + 1.8*np.sin(t/(SP/2)*2*np.pi + 0.7)
        + 0.004*t + rng.normal(0, .35, N))

ref = refs.materialize_iot(load, asset_id="chiller_6")

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(t, load, lw=.8); ax.axvspan(N-HOLD, N, color="crimson", alpha=.15)
ax.set(title="Chiller 6: 30 days hourly (last 24h held out, shaded)", xlabel="hour")
plt.tight_layout(); plt.show()
print(f"{N} points | TTM context 512 -> ~{N-512-96+1} training windows | forecasting {len(FH)}h")

## 3. The recipe contract

`recipe_template` tells an agent the shape without guessing. The `finetune` block is what makes the
training step configurable.

One gap worth knowing: **`save_to` works but `recipe_template` does not yet advertise it** - the cell
below checks. An agent relying on the template alone would not discover the persistence step, so it
is worth adding to `optional_blocks`.

In [ ]:
tmpl = M.recipe_template().model_dump()
print("estimator_spec:")
for line in tmpl["estimator_spec"]:
    print("  -", line[:112])
print("\noptional blocks (recipe_template lists these):")
for line in tmpl.get("optional_blocks", []):
    print("  -", line[:96])

# save_to is NOT yet listed by recipe_template - an agent would not discover it from the contract.
print("\n'finetune' in template:", any("finetune" in b for b in tmpl.get("optional_blocks", [])))
print("'save_to'  in template:", "save_to" in json.dumps(tmpl),
      " <- gap: works, but undocumented in the template")

## 4. The base card + preflight

A card is a pointer: `sktime_class` how to build it, `params` with what, `hf_repo` where the weights
live. `fit_strategy="zero-shot"` is pinned so the baseline is genuinely inference-only (sktime's
default `"minimal"` trains).

In [ ]:
BASE = {
    "model_id": "ttm_r2_base",
    "description": "IBM Granite TTM-R2, zero-shot. Baseline before adaptation.",
    "task_ids": ["tsfm_forecasting"],
    "sktime_class": "sktime.forecasting.ttm.TinyTimeMixerForecaster",
    "params": {"model_path": "ibm-granite/granite-timeseries-ttm-r2", "fit_strategy": "zero-shot"},
    "training_regime": "zero_shot",
    "hf_repo": "ibm-granite/granite-timeseries-ttm-r2",
    "context_length": 512, "prediction_length": 96,
    "provenance": "pretrained", "domain": "energy",
    "model_family": "TinyTimeMixer", "framework": "granite-tsfm",
    "tags": ["foundation", "zero-shot"],
}
print("register:", M.register_model(BASE).model_dump().get("status", "ok"))
d = M.resolve_model("ttm_r2_base").model_dump()
print("resolvable :", d.get("resolvable"), "| regime:", d.get("training_regime"))
print("reason     :", str(d.get("reason"))[:120])

## 5. Zero-shot baseline

We need a number to beat. `run_recipe` on the base card takes the zero-shot holdout path (one
rolling-origin split, no refit loop).

> **Needs torch.** Downloads TTM-R2 (~5MB) on first run.

In [ ]:
zs = M.run_recipe(dataset_path=ref, timestamp_column="timestamp", target_columns=["value"],
                  recipe={"estimator": {"model_id": "ttm_r2_base"}, "fh": FH,
                          "eval": {"metrics": ["mape"]}}).model_dump()
if "error" in zs:
    print("zero-shot baseline needs torch:", zs["error"][:100])
    ZS_MAPE = None
else:
    ZS_MAPE = zs["backtest_score"]
    print(f"zero-shot  regime={zs['training_regime']}  folds={zs.get('folds')}  MAPE={ZS_MAPE}")

## 6. Fine-tune AND save - one `run_recipe` call

The heart of it. The `finetune` block reaches the estimator (so it actually trains -
`fit_strategy="minimal"` adapts a parameter subset, the right choice for tiny data), and `save_to`
writes an HF-format checkpoint and returns its path in the result.

No `resolve()`, no `fit()`, no `save_pretrained()`. One tool call.

> **Needs torch.** Downloads weights, then trains.

In [ ]:
shutil.rmtree(CKPT, ignore_errors=True)
ft = M.run_recipe(
    dataset_path=ref, timestamp_column="timestamp", target_columns=["value"],
    recipe={"estimator": {"model_id": "ttm_r2_base"},
            "finetune": {"fit_strategy": "minimal",
                         "training_args": {"num_train_epochs": 3,
                                           "per_device_train_batch_size": 8,
                                           "learning_rate": 1e-4}},
            "save_to": CKPT,
            "fh": FH, "eval": {"metrics": ["mape"]}},
).model_dump()

if "error" in ft:
    print("fine-tune needs torch:", ft["error"][:110])
    print("On a torch box this returns:")
    print(f"   training_regime = fine_tune   (the block reached the estimator; it trained)")
    print(f"   trained         = True")
    print(f"   checkpoint_path = {CKPT}")
    CKPT_PATH = CKPT
    FT_MAPE = None
else:
    CKPT_PATH = ft["checkpoint_path"]
    FT_MAPE = ft["backtest_score"]
    print(f"regime          : {ft['training_regime']}  (fine_tune - it trained)")
    print(f"folds           : {ft.get('folds')}")
    print(f"backtest MAPE   : {FT_MAPE}")
    print(f"checkpoint_path : {CKPT_PATH}")
    print("files written   :", sorted(os.listdir(CKPT_PATH)) if os.path.isdir(CKPT_PATH) else "-")

## 7. Register the checkpoint

`register_finetuned` turns the checkpoint into a catalog card: it inherits the base's wrapper class,
points `params.model_path` at the checkpoint, records lineage, and **pins the serving regime to
`zero_shot`** - serving a fine-tune is load-and-predict, not train. (Unpinned, it would re-fine-tune
the already-fine-tuned weights on every serve.)

This cell runs with no torch - it is pure catalog bookkeeping.

In [ ]:
ftc = M.register_finetuned(
    model_id="ttm_r2_chiller6_ft", checkpoint_path=CKPT_PATH, base_model_id="ttm_r2_base",
    context_length=512, prediction_length=96,
    description="TTM-R2 fine-tuned on 30 days of Chiller 6 telemetry.",
    domain="energy",
).model_dump()
print("sktime_class    :", ftc.get("sktime_class"), "(inherited from base)")
print("params          :", ftc.get("params"))
print("provenance      :", ftc.get("provenance"), "(history: where the weights came from)")
print("training_regime :", ftc.get("training_regime"), "(next fit: load + predict, no training)")

lin = M.get_model_lineage("ttm_r2_chiller6_ft").model_dump()
print("lineage         :", lin.get("ancestors"), "root:", lin.get("root"))

## 8. Serve the fine-tuned model

Back to `run_recipe`, which neither knows nor cares that these weights were made 30 seconds ago. It
loads the checkpoint and forecasts.

> **Needs torch** to load the weights.

In [ ]:
sv = M.run_recipe(dataset_path=ref, timestamp_column="timestamp", target_columns=["value"],
                  recipe={"estimator": {"model_id": "ttm_r2_chiller6_ft"}, "fh": FH,
                          "eval": {"metrics": ["mape"]}}).model_dump()
if "error" in sv:
    print("serve needs torch:", sv["error"][:100])
    SV_MAPE = None
else:
    SV_MAPE = sv["backtest_score"]
    print(f"served  regime={sv['training_regime']}  folds={sv.get('folds')}  MAPE={SV_MAPE}")
    print("loaded YOUR checkpoint and predicted - no training on serve.")

## 9. Did the fine-tune help?

Both scored the same way (`zero_shot` holdout for the baseline, and the served fine-tune is
`zero_shot` too), so the two numbers are comparable.

In [ ]:
if ZS_MAPE is not None and SV_MAPE is not None:
    delta = (1 - SV_MAPE / ZS_MAPE) * 100
    print(f"  zero-shot   MAPE {ZS_MAPE:.4f}")
    print(f"  fine-tuned  MAPE {SV_MAPE:.4f}")
    print(f"  change      {delta:+.1f}%   (positive = fine-tune helped)")
    print("")
    print("  With a handful of training windows a small delta either way is noise. A negative")
    print("  number is a legitimate result: report it, do not re-roll until it wins.")
else:
    print("Comparison needs both runs (torch). On a torch box this prints both MAPEs and the delta.")
    print("Expect a modest change; ~90 windows of one chiller is not a lot to move an 805K-param")
    print("model pretrained on ~700M samples.")

## 10. The ledger

Every `run_recipe` is a run record. The baseline, the fine-tune, and the serve are all in
`list_runs`, so the decision is reproducible from the store alone.

In [ ]:
lr = M.list_runs().model_dump()
runs = lr.get("runs", [])
n = lr.get("count", len(runs))
if n == 0:
    print("0 runs recorded - the run_recipe calls above needed torch and did not execute here.")
    print("On a torch box you would see three: the zero-shot baseline, the fine-tune, and the serve,")
    print("each with its regime and score - the full decision reproducible from the store alone.")
else:
    print(f"{n} runs recorded")
    for r in runs[:10]:
        print(f"  {str(r.get('run_id')):18s} regime={str(r.get('training_regime')):13s} "
              f"score={r.get('backtest_score')}")

## What this shows

1. **The whole lifecycle is `run_recipe` + `register_finetuned`.** Fine-tune, save, register, serve -
   no dropping to `resolve()`/`fit()`/`save_pretrained()` by hand. That is what the `finetune` block
   reaching the estimator and `save_to` persisting the weights buys you.
2. **`save_to` returns the `checkpoint_path`** that `register_finetuned` consumes. The two calls
   snap together; nothing in between is manual.
3. **`provenance` vs `training_regime`.** The checkpoint card is `provenance=finetuned` (history) but
   `training_regime=zero_shot` (serving loads and predicts). `register_finetuned` pins this so
   serving does not silently re-train.
4. **Fine-tuning is not automatically better.** ~90 windows of one asset against a model pretrained
   on ~700M samples - a small or negative delta is a real outcome. Score both the same way and
   report honestly.
5. **`fit_strategy="minimal"` is the tiny-data setting**, and tiny still means hundreds of points.
   With 40, skip the fine-tune and serve TTM zero-shot.